### Statistical and Grid network package

In [50]:
import os
import numpy as np
from tqdm import tqdm
import pandas as pd
import pandapower as pp
from pandapower.powerflow import LoadflowNotConverged
import networkx as nx
import pandapower.networks as pn
import pandapower.plotting as plot
from pandapower.control import ConstControl
from pandapower.timeseries import DFData, OutputWriter, run_timeseries
import matplotlib.pyplot as plt
from scipy.stats import norm
from dowhy import CausalModel
from pandapower.pypower.makeYbus import makeYbus
import scipy.linalg
import scipy.sparse as sp

### Define the Function to check N-1 contingency criterion with the test case

In [51]:
# Define the function to check whether the network meet the N-1 contingency criterion
def check_n_1_contingency(net):
    critical_elements = []
    
    # Backup original network
    original_net = net.deepcopy()
    
    # Test line outages
    for line in net.line.index:
        net_copy = original_net.deepcopy()
        net_copy.line.at[line, "in_service"] = False  # Remove one line
        try:
            pp.runpp(net_copy, algorithm="nr")
        except pp.powerflow.LoadflowNotConverged:
            critical_elements.append(f"Line {line} outage causes failure.")
            continue
        
        # Check for overloads
        if any(net_copy.res_line.loading_percent > 100):
            critical_elements.append(f"Line {line} outage causes overloads.")

    # Test generator outages
    for gen in net.gen.index:
        net_copy = original_net.deepcopy()
        net_copy.gen.at[gen, "in_service"] = False  # Remove one generator
        try:
            pp.runpp(net_copy, algorithm="nr")
        except pp.powerflow.LoadflowNotConverged:
            critical_elements.append(f"Generator {gen} outage causes failure.")
            continue

        # Check for voltage violations
        if any((net_copy.res_bus.vm_pu < 0.95) | (net_copy.res_bus.vm_pu > 1.05)):
            critical_elements.append(f"Generator {gen} outage causes voltage issues.")

    return critical_elements

#### Load the network case

In [52]:
# Load the IEEE 24-bus reliability test system 
net = pn.case24_ieee_rts()
pp.runpp(net)

In [53]:
# Tag buses for reference in the information layer
for i in net.bus.index:
    net.bus.at[i, "tag"] = f"Bus-{i}"

### The Function to reinforce the network to meet the N-1 criterion

In [54]:
# Function to reinforce the network based on contingency violations
def reinforce_network(net):
    original_net = net.deepcopy()
    
    # Identify problematic lines
    overloaded_lines = []
    for line in net.line.index:
        net_copy = original_net.deepcopy()
        net_copy.line.at[line, "in_service"] = False  # Simulate line outage
        try:
            pp.runpp(net_copy)
        except pp.powerflow.LoadflowNotConverged:
            overloaded_lines.append(line)
            continue
        if any(net_copy.res_line.loading_percent > 100):
            overloaded_lines.append(line)
    
    # Add parallel lines to overloaded lines
    for line in overloaded_lines:
        from_bus = net.line.loc[line, "from_bus"]
        to_bus = net.line.loc[line, "to_bus"]
        print(f"Adding parallel line between Bus {from_bus} and Bus {to_bus} to mitigate overload.")
        pp.create_line_from_parameters(net, from_bus=from_bus, to_bus=to_bus, 
                                       length_km=1.0, r_ohm_per_km=0.05, 
                                       x_ohm_per_km=0.1, c_nf_per_km=0, 
                                       max_i_ka=1.5)  # Higher capacity
    
    # Identify problematic generators
    problematic_gens = []
    for gen in net.gen.index:
        net_copy = original_net.deepcopy()
        net_copy.gen.at[gen, "in_service"] = False  # Simulate generator outage
        try:
            pp.runpp(net_copy)
        except pp.powerflow.LoadflowNotConverged:
            problematic_gens.append(gen)
            continue
        if any((net_copy.res_bus.vm_pu < 0.95) | (net_copy.res_bus.vm_pu > 1.05)):
            problematic_gens.append(gen)

    # Increase generator capacities to provide redundancy
    for gen in problematic_gens:
        net.gen.at[gen, "p_mw"] *= 1.2  # Increase power generation by 20%
        print(f"Increasing capacity of Generator {gen} to improve voltage stability.")

    return net

# Reinforce the network
net = reinforce_network(net)

# Run contingency check again
pp.runpp(net)
print("Reinforced system power flow successful.")

# Check if violations still exist
violations = check_n_1_contingency(net)
if violations:
    print("System still has issues under N-1 contingency.")
    for v in violations:
        print(v)
else:
    print("The RTS 24-bus system satisfies the N-1 contingency criterion!")

Adding parallel line between Bus 1 and Bus 5 to mitigate overload.
Adding parallel line between Bus 5 and Bus 9 to mitigate overload.
Increasing capacity of Generator 0 to improve voltage stability.
Increasing capacity of Generator 2 to improve voltage stability.
Increasing capacity of Generator 3 to improve voltage stability.
Increasing capacity of Generator 5 to improve voltage stability.
Increasing capacity of Generator 6 to improve voltage stability.
Increasing capacity of Generator 8 to improve voltage stability.
Increasing capacity of Generator 9 to improve voltage stability.
Reinforced system power flow successful.
System still has issues under N-1 contingency.
Line 15 outage causes overloads.
Line 16 outage causes overloads.
Line 21 outage causes overloads.
Line 23 outage causes overloads.
Generator 1 outage causes voltage issues.
Generator 2 outage causes voltage issues.
Generator 3 outage causes voltage issues.
Generator 4 outage causes voltage issues.
Generator 5 outage caus

In [55]:
import copy
import numpy as np
import pandapower as pp

def check_n_1_contingency_pf(net, vmin=0.95, vmax=1.05, line_loading_limit=100.0):
    """
    N-1 check using power flow (fixed dispatch).
    Returns a list of human-readable violation strings and a structured dict.
    """
    critical = []
    details = {"line_outage": [], "gen_outage": []}

    original_net = copy.deepcopy(net)  

    # ---- Line outages ----
    for line in original_net.line.index:
        net_copy = copy.deepcopy(original_net)
        net_copy.line.at[line, "in_service"] = False
        try:
            pp.runpp(net_copy, algorithm="nr")
        except pp.powerflow.LoadflowNotConverged:
            critical.append(f"Line {line} outage causes failure (PF not converged).")
            details["line_outage"].append({"outaged_line": int(line), "reason": "pf_not_converged"})
            continue

        # survivor line overloads
        over_mask = net_copy.res_line.loading_percent > line_loading_limit
        overloaded = list(net_copy.res_line.index[over_mask])
        if line in overloaded:
            overloaded.remove(line)

        vm = net_copy.res_bus.vm_pu.values
        v_ok = (vm.min() >= vmin) and (vm.max() <= vmax)

        if overloaded or not v_ok:
            msg = []
            if overloaded:
                msg.append(f"overloads on {overloaded}")
            if not v_ok:
                msg.append(f"voltage min={vm.min():.3f}, max={vm.max():.3f}")
            critical.append(f"Line {line} outage -> " + "; ".join(msg))
            details["line_outage"].append({
                "outaged_line": int(line),
                "overloaded_lines": overloaded,
                "min_vm": float(vm.min()),
                "max_vm": float(vm.max()),
                "reason": "limits"
            })

    # ---- Gen outages ----
    for g in original_net.gen.index:
        net_copy = copy.deepcopy(original_net)
        net_copy.gen.at[g, "in_service"] = False
        try:
            pp.runpp(net_copy, algorithm="nr")
        except pp.powerflow.LoadflowNotConverged:
            critical.append(f"Generator {g} outage causes failure (PF not converged).")
            details["gen_outage"].append({"outaged_gen": int(g), "reason": "pf_not_converged"})
            continue

        vm = net_copy.res_bus.vm_pu.values
        v_ok = (vm.min() >= vmin) and (vm.max() <= vmax)
        if not v_ok:
            critical.append(f"Generator {g} outage causes voltage issues (min={vm.min():.3f}, max={vm.max():.3f}).")
            details["gen_outage"].append({
                "outaged_gen": int(g),
                "min_vm": float(vm.min()),
                "max_vm": float(vm.max()),
                "reason": "voltage"
            })

        over_mask = net_copy.res_line.loading_percent > line_loading_limit
        overloaded = list(net_copy.res_line.index[over_mask])
        if overloaded:
            critical.append(f"Generator {g} outage causes line overloads: {overloaded}.")
            # merge with existing record if any
            found = next((d for d in details["gen_outage"] if d.get("outaged_gen")==int(g)), None)
            if found:
                found["overloaded_lines"] = overloaded
                found["reason"] = "limits"
            else:
                details["gen_outage"].append({
                    "outaged_gen": int(g),
                    "overloaded_lines": overloaded,
                    "reason": "limits"
                })

    details["ok"] = (len(critical) == 0)
    return critical, details


### Built PMU with the Loaded Network Grid

In [15]:
def get_single_bus_pmu_measurement(net, bus_index):
    """
    Extract PMU measurement for a single bus:
    - Voltage magnitude (p.u.)
    - Voltage angle (degrees)
    - Active power injection (MW)
    - Reactive power injection (MVAR)
    
    Returns a dictionary with all relevant fields.
    """
    vm = float(net.res_bus.vm_pu.at[bus_index])
    va = float(net.res_bus.va_degree.at[bus_index])
    p = float(net.res_bus.p_mw.at[bus_index])
    q = float(net.res_bus.q_mvar.at[bus_index])
    
    return {
        'bus': bus_index,
        'vm_pu': vm,
        'va_degree': va,
        'p_mw': p,
        'q_mvar': q
    }


def pdc(net, pmu_buses):
    """
    Simulate the Phasor Data Concentrator (PDC):
    Collects PMU data from all buses listed in `pmu_buses`.
    Returns a list of measurement dictionaries.
    """
    pmu_data = []
    for bus in pmu_buses:
        measurement = get_single_bus_pmu_measurement(net, bus)
        pmu_data.append(measurement)
    return pmu_data


# Example usage
pmu_buses = net.bus.index.tolist()  # simulate PMU on all buses
pmu_data_frame = pdc(net, pmu_buses)

# Optional: convert to a table (like your screenshot)
import pandas as pd
pmu_df = pd.DataFrame(pmu_data_frame)
print(pmu_df)


    bus     vm_pu  va_degree       p_mw     q_mvar
0     0  1.000000   0.000000  -1.651633  -3.281965
1     1  1.000000   0.001457 -51.464000 -13.368121
2     2  0.999964  -0.001080   2.400000   1.200000
3     3  0.999936  -0.001438   7.600000   1.600000
4     4  0.999932  -0.002318   0.000000  -0.189974
5     5  0.999894  -0.003741   0.000000   0.000000
6     6  0.999855  -0.005822  22.800000  10.900000
7     7  0.999773  -0.006256  30.000000  30.000000
8     8  0.999896  -0.004551   0.000000   0.000000
9     9  0.999897  -0.005358   5.800000   2.000000
10   10  0.999896  -0.004551   0.000000   0.000000
11   11  0.999909  -0.000082  11.200000   7.500000
12   12  1.000000   0.011795 -37.000000   1.776444
13   13  0.999897  -0.002394   6.200000   1.600000
14   14  0.999910  -0.003013   8.200000   2.500000
15   15  0.999873  -0.003023   3.500000   1.800000
16   16  0.999857  -0.005148   9.000000   5.800000
17   17  0.999872  -0.005614   3.200000   0.900000
18   18  0.999847  -0.007350   

In [13]:
 net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.000000,0.000000,-1.651633,-3.281965
1,1.000000,0.001457,-51.464000,-13.368121
2,0.999964,-0.001080,2.400000,1.200000
3,0.999936,-0.001438,7.600000,1.600000
4,0.999932,-0.002318,0.000000,-0.189974
5,0.999894,-0.003741,0.000000,0.000000
6,0.999855,-0.005822,22.800000,10.900000
7,0.999773,-0.006256,30.000000,30.000000
8,0.999896,-0.004551,0.000000,0.000000
9,0.999897,-0.005358,5.800000,2.000000


### Information Layer Construct

In [ ]:
# Define sensors at selected buses
pmu_buses = [1, 3, 7, 15]
sensor_data = {}

for bus in pmu_buses:
    sensor_data[f"Bus-{bus}"] = {
        "type": "PMU",
        "vm_pu_true": None,
        "vm_pu_measured": None,
        "measurement_delay": 1,  # in timesteps
        "communication_error": 0.01  # 1% measurement noise
    }


In [ ]:
def simulate_info_layer(net, sensor_data):
    for bus_tag, data in sensor_data.items():
        bus_idx = int(bus_tag.split("-")[1])
        true_value = net.res_bus.vm_pu.at[bus_idx]
        noise = np.random.normal(0, data["communication_error"])
        sensor_data[bus_tag]["vm_pu_true"] = true_value
        sensor_data[bus_tag]["vm_pu_measured"] = true_value + noise

### Apply the cyber attack by injecting false data

In [ ]:
def inject_false_data(pmu_data, attack_buses, voltage_offset=0.05, angle_offset=2.0):
    """
    Inject false data at specified bus indices.
    voltage_offset: additive error in per unit (p.u.)
    angle_offset: additive error in degrees
    """
    pmu_data_attacked = pmu_data.copy()
    # Create copies to avoid modifying original
    voltage_attacked = np.copy(pmu_data_attacked['voltage'])
    angle_attacked = np.copy(pmu_data_attacked['angle'])
    
    for idx in attack_buses:
        voltage_attacked[idx] += voltage_offset  # add offset
        angle_attacked[idx] += angle_offset
    pmu_data_attacked['voltage'] = voltage_attacked
    pmu_data_attacked['angle'] = angle_attacked
    return pmu_data_attacked

# Assume attackers compromise bus 5, 12, and 23 
attack_buses = [5, 12, 23]
pmu_data_attacked = inject_false_data(pmu_data_true, attack_buses)

Sample Metric: Voltage Deviation Post-Attack

In [ ]:
def compute_voltage_deviation(net, baseline_vm_pu):
    deviations = abs(net.res_bus.vm_pu - baseline_vm_pu)
    return deviations.mean()

In [ ]:
def compute_custom_resilience_metric(system_loss, recovery_time, max_loss=1.0):
    # Normalize values between 0 and 1
    L_norm = system_loss / max_loss
    T_norm = recovery_time / 10  # assume 10 is worst case

    # Resilience = 1 - (weighted penalty)
    return 1 - (0.6 * L_norm + 0.4 * T_norm)

### PDC Mode

### The Resilience Metric

### Evaluate resilience via:

1.Time to recovery

2. Extent of degraded performance

3. Cascading failure depth

4. Resilience Index (NEDI, R_cyber, etc.)

### Run Time-Series Simulations with Events

Define multiple time steps to simulate pre-attack, attack, recovery periods.

In [ ]:
T = 10
baseline_vm_pu = None

for t in range(T):
    if t == 0:
        pp.runpp(net)
        baseline_vm_pu = net.res_bus.vm_pu.copy()

    if t == 3:
        apply_fdia(sensor_data, target_bus=7, offset=0.15)

    if t == 5:
        break_communication(sensor_data, bus_id=15)

    simulate_info_layer(net, sensor_data)

    # Log metrics
    voltage_dev = compute_voltage_deviation(net, baseline_vm_pu)
    print(f"t={t}, Voltage Deviation: {voltage_dev:.4f}")
